# Лабораторна робота 3 — Градієнтний спуск для множинної регресії

**Набір даних:** `kc_house_data.csv`  
**Обмеження:** scikit-learn-регресія **не дозволена** для базових завдань.

## Налаштування

In [1]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt
from sklearn.model_selection import train_test_split

%matplotlib inline


## Теоретичне підґрунтя

Градієнт RSS відносно ваги *wᵢ*:
```
gradient_i = 2 · dot(errors, feature_i)     де  errors = Xw − y
```
Зупинка відбувається, коли ‖gradient‖₂ < `tolerance`.

---
## Завдання 1 — Підготовка даних та `get_numpy_data()`

Завантажте `kc_house_data.csv`. Розбийте **20 % навчання / 80 % тест** (`random_state=0`).

Реалізуйте `get_numpy_data(dataframe, features, output)`, яка:
1. Додає стовпець з одиницями ліворуч від матриці ознак (для вільного члена).
2. Повертає `(feature_matrix, output_array)` як масиви NumPy.

In [9]:
sales = pd.read_csv('kc_house_data.csv')
train_data, test_data = train_test_split(sales, test_size=0.8, random_state=0)
sales.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [4]:
def get_numpy_data(dataframe, features, output):
    """
    Будує матрицю ознак NumPy (зі стовпцем вільного члена) та вектор виходу.

    Параметри
    ----------
    dataframe : pd.DataFrame
    features  : список str   назви стовпців-ознак
    output    : str          назва стовпця цільової змінної

    Повертає
    -------
    feature_matrix : np.ndarray, shape (n, len(features)+1)
    output_array   : np.ndarray, shape (n,)
    """
    df = dataframe.copy()
    df['constant'] = 1.0
    features_with_constant = ['constant'] + features
    feature_matrix = df[features_with_constant].to_numpy(dtype=float)
    output_array = dataframe[output].to_numpy(dtype=float)
    
    return feature_matrix, output_array


### Перевірка

In [5]:
example_features, example_output = get_numpy_data(sales, ['sqft_living'], 'price')
print('Перший рядок матриці ознак:', example_features[0, :])  # має бути [1.0, <sqft>]
print('Перше значення виходу:    ', example_output[0])


Перший рядок матриці ознак: [1.00e+00 1.18e+03]
Перше значення виходу:     221900.0


---
## Завдання 2 — Реалізація `predict_output()`

Використовуйте **один виклик `np.dot`** — без явних циклів.

In [11]:
def predict_output(feature_matrix, weights):
    """
    Обчислює передбачення як скалярний добуток кожного рядка з вагами.

    Повертає
    -------
    predictions : np.ndarray, shape (n,)
    """
    return np.dot(feature_matrix, weights)


### Перевірка — перші два передбачення мають бути ≈1181 і ≈2571

In [12]:
my_weights = np.array([1.0, 1.0])
test_preds = predict_output(example_features, my_weights)
print(f'передбачення[0]: {test_preds[0]:.1f}  (очікується ≈1181)')
print(f'передбачення[1]: {test_preds[1]:.1f}  (очікується ≈2571)')


передбачення[0]: 1181.0  (очікується ≈1181)
передбачення[1]: 2571.0  (очікується ≈2571)


---
## Завдання 3 — Реалізація `feature_derivative()`

Похідна RSS відносно однієї ваги — це `2 · dot(errors, feature)`. Реалізуйте як один вираз NumPy.

In [13]:
def feature_derivative(errors, feature):
    """Повертає похідну RSS відносно ваги для даної ознаки."""
    return 2 * np.dot(errors, feature);


### Перевірка — похідна відносно константної ознаки має дорівнювати `-2 · sum(prices)`

In [14]:
zero_weights = np.array([0.0, 0.0])
zero_preds   = predict_output(example_features, zero_weights)
errors       = zero_preds - example_output              # = -prices when weights=0

deriv_constant = feature_derivative(errors, example_features[:, 0])
expected       = -2 * np.sum(example_output)
print(f'Обчислено: {deriv_constant:.2e}')
print(f'Очікується: {expected:.2e}')
print(f'Збіг: {np.isclose(deriv_constant, expected)}')


Обчислено: -2.33e+10
Очікується: -2.33e+10
Збіг: True


---
## Завдання 4 — Градієнтний спуск

Реалізуйте `regression_gradient_descent(feature_matrix, output, initial_weights, step_size, tolerance)`. Функція оновлює всі ваги одночасно на кожній ітерації та повертає їх, коли ‖gradient‖₂ < `tolerance`.

Запустіть з такими параметрами на **навчальних** даних:
```
features        = ['sqft_living']
initial_weights = [-47000., 1.]
step_size       = 7e-12
tolerance       = 2.5e7
```
Вкажіть навчені ваги. Яка передбачувана ціна першого будинку в **тестовій** вибірці? Обчисліть RSS на всій тестовій вибірці.

In [15]:
def regression_gradient_descent(feature_matrix, output,
                                initial_weights, step_size, tolerance):
    """
    Мінімізує RSS за допомогою пакетного градієнтного спуску.

    Повертає
    -------
    weights : np.ndarray — навчені ваги
    """
    weights   = np.array(initial_weights, dtype=float)
    converged = False

    while not converged:
        predictions = predict_output(feature_matrix, weights)
        errors = predictions - output

        gradient_sum_squares = 0.0
        for i in range(len(weights)):
            derivative = feature_derivative(errors, feature_matrix[:, i])
            gradient_sum_squares += (derivative ** 2)
            weights[i] -= (step_size * derivative)

        gradient_magnitude = sqrt(gradient_sum_squares)
        if gradient_magnitude < tolerance:
            converged = True

    return weights


### Запуск Моделі 1 — одна ознака

In [19]:
simple_feature_matrix, output = get_numpy_data(train_data, ['sqft_living'], 'price')

simple_weights = regression_gradient_descent(
    simple_feature_matrix, output,
    initial_weights=[-47000., 1.],
    step_size=7e-12,
    tolerance=2.5e7
)
print('Навчені ваги:', simple_weights)


Навчені ваги: [-46999.88018846    280.51005013]


### Оцінка Моделі 1 на тестовій вибірці

In [20]:
test_feature_matrix, test_output = get_numpy_data(test_data, ['sqft_living'], 'price')

# Predicted price for the first test house
test_predictions = predict_output(test_feature_matrix, simple_weights)
print(f'Передбачена ціна (будинок 0): ${test_predictions[0]:,.0f}')
print(f'Реальна ціна    (будинок 0): ${test_output[0]:,.0f}')

# RSS on the full test set
rss_model1 = np.sum((test_predictions - test_output) ** 2)
print(f'Тестова RSS Моделі 1: {rss_model1:.3e}')


Передбачена ціна (будинок 0): $354,129
Реальна ціна    (будинок 0): $297,000
Тестова RSS Моделі 1: 1.204e+15


---
## ✨ Бонус — Додайте другу ознаку

Повторіть градієнтний спуск з `['sqft_living', 'sqft_living15']` та параметрами:
```
initial_weights = [-100000., 1., 1.]
step_size       = 4e-12
tolerance       = 1e9
```
Обчисліть RSS на тестовій вибірці та порівняйте з моделлю з однією ознакою. Порівняйте передбачену ціну першого тестового будинку з обох моделей з реальною ціною.

In [ ]:
# Бонус — модель з двома ознаками
# Параметри: initial_weights=[-100000., 1., 1.], step_size=4e-12, tolerance=1e9

model2_features = ['sqft_living', 'sqft_living15']
feature_matrix_train, output_train = get_numpy_data(train_data, model2_features, 'price')

initial_weights = np.array([-100000., 1., 1.])
step_size = 4e-12
tolerance = 1e9

weights_model2 = regression_gradient_descent(
    feature_matrix_train, output_train, 
    initial_weights, step_size, tolerance
)

feature_matrix_test, output_test = get_numpy_data(test_data, model2_features, 'price')
predictions_model2 = predict_output(feature_matrix_test, weights_model2)

print(f"Навчені ваги (Модель 2): {weights_model2}")
print(f"Передбачена ціна (будинок 0): ${predictions_model2[0]:,.0f}")
print(f"Реальна ціна    (будинок 0): ${output_test[0]:,.0f}")

rss_model2 = np.sum((predictions_model2 - output_test) ** 2)
print(f"Тестова RSS Моделі 2: {rss_model2:.3e}")

print(f"Різниця RSS (M1 - M2): {rss_model1 - rss_model2:.3e}")

Навчені ваги (Модель 2): [-9.99998826e+04  2.43610628e+02  6.54855462e+01]
Передбачена ціна (будинок 0): $342,008
Реальна ціна    (будинок 0): $297,000
Тестова RSS Моделі 2: 1.186e+15
Різниця RSS (M1 - M2): 1.780e+13
